# VLM Serving Platform — Kaggle Remote Worker (Phase 2)

This notebook turns a **Kaggle GPU (Tesla T4 / P100)** into an outbound remote worker serving Vision-Language Models (e.g., `Qwen2-VL-2B-Instruct` or `Qwen2-VL-7B-Instruct`).

### Why Outbound WebSocket?
- Kaggle blocks all incoming network ports (no public IP, no ingress).
- The worker establishes an **outbound persistent WebSocket connection** to your local/cloud Gateway through a **Cloudflare Tunnel** (`wss://your-subdomain.trycloudflare.com/workers/connect`).
- Inference jobs dispatched by Gateway are processed locally on the Kaggle GPU, and results stream back across the WebSocket connection.

In [ ]:
# 1. Install required dependencies
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate websockets pydantic pillow requests

In [ ]:
# 2. Clone or mount repository code
import os, sys
repo_dir = '/kaggle/working/vlm-serving-platform'
if not os.path.exists(repo_dir):
    # Clone your GitHub repository or use uploaded Kaggle dataset
    !git clone https://github.com/your-username/vlm-serving-platform.git {repo_dir}

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

In [ ]:
# 3. Verify CUDA availability and GPU properties
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# 4. Load Qwen2-VL Model Adapter
from packages.model_adapters.qwen_vl import QwenVLAdapter

model_id = "Qwen/Qwen2-VL-2B-Instruct"
print(f"Loading {model_id} on GPU...")
adapter = QwenVLAdapter(
    model_id=model_id,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    torch_dtype="bfloat16" if torch.cuda.is_bf16_supported() else "float16",
    max_pixels=1280 * 28 * 28,
    min_pixels=256 * 28 * 28,
)
print(f"Model {model_id} loaded successfully!")

In [ ]:
# 5. Connect outbound to Gateway and serve inference jobs
import asyncio
from packages.worker_runtime.remote_client import RemoteWorkerClient

# Set your Cloudflare Tunnel WebSocket URL and shared secret token
GATEWAY_WS_URL = "wss://your-tunnel-subdomain.trycloudflare.com/workers/connect"
AUTH_TOKEN = "vlm_worker_secret_2026"
WORKER_ID = "kaggle_t4_qwen2vl_01"

client = RemoteWorkerClient(
    gateway_ws_url=GATEWAY_WS_URL,
    adapter=adapter,
    auth_token=AUTH_TOKEN,
    worker_id=WORKER_ID,
)

print(f"Starting remote worker '{WORKER_ID}' connecting to {GATEWAY_WS_URL}...")
try:
    await client.run()
except (KeyboardInterrupt, asyncio.CancelledError):
    print("Worker stopped.")